In [1]:
# =============================================================================
#  Relations Need a Home — interactive demo
#  Haziq Mohammad Khalid, Hamad Siddiqi | COE486
#
#  ONE CELL. Paste into Colab / Kaggle / Jupyter and run. Or: python demo.py
#
#  Type a CV-Bench Relation index, get:
#     - the two objects localised from the frozen model's OWN attention
#     - the vanilla answer
#     - all five position-injection deliveries (Table 2)
#     - our relation-token interface (Table 3)
#  Results stream in as each condition finishes.
#
#  REQUIRES A GPU. LLaVA-1.5-7B is ~14 GB in fp16.
# =============================================================================

import os, sys, subprocess

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

try:
    import gradio, torch, transformers, datasets, huggingface_hub, matplotlib
except ImportError:
    _pip("gradio>=4.44", "torch", "transformers>=4.39", "datasets",
         "huggingface_hub", "matplotlib", "accelerate", "sentencepiece", "protobuf")

import re, io, math, json, warnings
import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
import gradio as gr
from huggingface_hub import hf_hub_download
from transformers import LlavaForConditionalGeneration, AutoProcessor
from datasets import load_dataset

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------- config ---
HF_REPO      = "Haziq-exe/relational-binding-interface"
CKPT_FILE    = "abstractor_relation_kl_ce.pt"
SPLIT_FILE   = "relation_split.json"          # optional; used to label train/test
LLAVA_ID     = "llava-hf/llava-1.5-7b-hf"
GRID, N_VIS  = 24, 576
LETTERS      = "ABCDEFGHIJ"
PROMPT_TEMPLATE = "USER: <image>\n{q} ASSISTANT:"
MCQ_SUFFIX      = "\nAnswer with the letter of the correct option only."
ANS_MAX_NEW     = 8

# Localisation config — MUST match RES_LOC / LOC_KW in the training notebook,
# so the demo shows the same coordinates every reported number was computed from.
LOC_KW = dict(layers=(14,), rel_att=True, denoise=0.75, smooth=False,
              token_reduce="mean", locate="centroid")
# Steering config — the swept values from the paper (Sec. 3).
STEER  = dict(layer=14, beta_abs=2.75, beta_rel=0.7, magnitude="norm",
              sigma_scale=False, whiten=False, rich_pairs=True, orthogonalize=True)

# ------------------------------------------------------------ GPU is a must --
if not torch.cuda.is_available():
    raise SystemExit(
        "\n" + "=" * 72 +
        "\n  NO GPU DETECTED — this demo cannot run.\n"
        "\n  It loads a frozen LLaVA-1.5-7B (~14 GB in fp16). CPU inference would\n"
        "  take many minutes per answer and most likely exhaust system RAM.\n"
        "\n  Colab  : Runtime > Change runtime type > T4 GPU (or better)\n"
        "  Kaggle : Settings > Accelerator > GPU T4 x2\n"
        "  Local  : a CUDA GPU with >=16 GB, or >=24 GB to be comfortable\n"
        + "=" * 72 + "\n")

_gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
_vram = sum(torch.cuda.get_device_properties(i).total_memory
            for i in range(torch.cuda.device_count())) / 1e9
print(f"GPU: {', '.join(_gpu_names)}  |  total VRAM ~{_vram:.0f} GB")
DTYPE = torch.float16          # T4 has no bf16

# ------------------------------------------------------------- load model ---
print("Loading LLaVA-1.5-7B (frozen)…")
processor = AutoProcessor.from_pretrained(LLAVA_ID)
model = LlavaForConditionalGeneration.from_pretrained(
    LLAVA_ID, torch_dtype=DTYPE, device_map="auto", low_cpu_mem_usage=True, attn_implementation="eager")
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

def _cfg_get(*names, default=None, required=True):
    for src in (model.config, getattr(model.config, "text_config", None),
                getattr(model.config, "vision_config", None)):
        if src is None:
            continue
        for n in names:
            if hasattr(src, n):
                return getattr(src, n)
    if required and default is None:
        raise AttributeError(f"none of {names} on model.config")
    return default

IMAGE_TOKEN_INDEX = _cfg_get("image_token_index", "image_token_id")
HIDDEN            = _cfg_get("hidden_size")

def _resolve_layers(m):
    for path in ("language_model.model.layers", "model.language_model.layers",
                 "language_model.layers", "model.layers"):
        cur = m
        try:
            for part in path.split("."):
                cur = getattr(cur, part)
            if isinstance(cur, (nn.ModuleList, list)) and len(cur) > 1:
                return cur
        except AttributeError:
            continue
    for mod in m.modules():                       # structural fallback
        if isinstance(mod, nn.ModuleList) and len(mod) >= 24:
            return mod
    raise RuntimeError("could not locate the decoder layer stack")

LM_LAYERS = _resolve_layers(model)
PRIMARY_DEVICE = next(model.get_input_embeddings().parameters()).device

# geometry assertion: <image> MUST expand to exactly 576 tokens, or the 24x24
# map-to-token correspondence the whole method rests on is broken.
_probe = processor(images=Image.new("RGB", (336, 336)),
                   text=PROMPT_TEMPLATE.format(q="test"), return_tensors="pt")
_n_img = int((_probe["input_ids"][0] == IMAGE_TOKEN_INDEX).sum())
assert _n_img == N_VIS, f"expected {N_VIS} visual tokens, got {_n_img}"
print(f"geometry OK: <image> -> {_n_img} tokens | hidden={HIDDEN} | layers={len(LM_LAYERS)}")

# ------------------------------------------------------------ CV-Bench -------
print("Loading CV-Bench…")
cvbench = load_dataset("nyu-visionx/CV-Bench", split="test")

def _strip_boxes(q):
    return re.sub(r"\s*\((?:annotated|highlighted)[^)]*\)", "", str(q)).strip()

def _relation_objects(clean_q, target_class=None):
    q = (clean_q or "").strip()
    for pat in (r"relative positions of the (.+?) and the (.+?)(?:\s+in the image|,|\.|\?|$)",
                r"where is the (.+?) (?:located |positioned )?with respect to the (.+?)[?.]?\s*$",
                r"is the (.+?) (?:to the )?(?:left|right|above|below)\s+(?:of\s+)?the (.+?)[?.]?\s*$"):
        m = re.search(pat, q, re.I)
        if m and m.group(1).strip() and m.group(2).strip():
            return [m.group(1).strip(), m.group(2).strip()]
    return [target_class] if target_class else []

def _relation_phrases(question):
    q = _strip_boxes(question)
    m = re.search(r"positions of the (.+?) and the (.+?)"
                  r"(?: in the image| in the picture|,|\?| where| located| with respect)", q, re.I)
    if not m:
        m = re.search(r"where is the (.+?) (?:located |positioned )?with respect to the (.+?)[\?\.]", q, re.I)
    return [m.group(1).strip(), m.group(2).strip()] if m else []

def _match_phrase(obj, phrases):
    o = obj.lower().strip()
    for ph in phrases:
        if o in ph.lower():
            return ph
    for ph in phrases:
        if o.split()[-1] in ph.lower():
            return ph
    return obj

def _view(i):
    r = cvbench[int(i)]
    base = r.get("prompt") or r["question"]
    return {"i": int(i), "image": r["image"], "question": base + MCQ_SUFFIX,
            "clean_question": _strip_boxes(r["question"]), "choices": list(r["choices"]),
            "answer": r["answer"], "task": r.get("task"),
            "target_class": r.get("target_class")}

REL_IDX = [i for i, t in enumerate(cvbench["task"]) if t == "Relation"]
print(f"CV-Bench: {len(cvbench)} rows | {len(REL_IDX)} Relation items")

# optional split labels
SPLIT = None
try:
    SPLIT = json.load(open(hf_hub_download(HF_REPO, SPLIT_FILE, repo_type="model")))
    print(f"split loaded: train={len(SPLIT['train'])} val={len(SPLIT['val'])} test={len(SPLIT['test'])}")
except Exception:
    print("note: relation_split.json not found on the Hub — train/test labels disabled")

def _split_of(rel_pos):
    if not SPLIT:
        return None
    for name in ("train", "val", "test"):
        if rel_pos in SPLIT[name]:
            return name
    return None

# ------------------------------------------------- attention -> coordinates --
def _find_subseq(hay, needle, start=0):
    n = len(needle)
    if n == 0:
        return -1
    for i in range(start, len(hay) - n + 1):
        if hay[i:i + n] == needle:
            return i
    return -1

def find_noun_positions(ids, noun, tokenizer, search_start=0):
    for variant in (" " + noun, noun, " " + noun.lower(), noun.lower()):
        sub = tokenizer(variant, add_special_tokens=False).input_ids
        j = _find_subseq(ids, sub, search_start)
        if j >= 0:
            return list(range(j, j + len(sub)))
    target = " ".join(noun.lower().split())
    pieces = [tokenizer.decode([t]) for t in ids]
    for i in range(max(search_start, 0), len(ids)):
        acc = ""
        for k in range(i, len(ids)):
            acc += pieces[k]
            norm = " ".join(acc.lower().split())
            if norm == target:
                return list(range(i, k + 1))
            if not target.startswith(norm):
                break
    return []

def _baseline_map(attentions, text_positions, vis_idx, layers):
    accs = []
    for L in layers:
        t = attentions[L][0][:, text_positions, :][:, :, vis_idx]
        accs.append(t.mean(0).mean(0).float())
    return torch.stack(accs, 0).mean(0).reshape(GRID, GRID).cpu().numpy()

def _noun_maps(attentions, positions, vis_idx, layers):
    out = []
    for p in positions:
        per_layer = [attentions[L][0][:, p, :][:, vis_idx].mean(0).float() for L in layers]
        out.append(torch.stack(per_layer, 0).mean(0))
    return torch.stack(out, 0).reshape(len(positions), GRID, GRID).cpu().numpy()

def _denoise(m, q):
    m = np.asarray(m, float)
    return np.clip(m - np.quantile(m, q), 0, None) if q else m

def _coords(m):
    m = np.asarray(m, float)
    w = m.sum()
    if w <= 0:
        return None, None
    rows, cols = np.indices(m.shape)
    cx = float((cols * m).sum() / w) / (GRID - 1)
    cy = 1.0 - float((rows * m).sum() / w) / (GRID - 1)   # row 0 is top -> y=1
    return cx, cy

@torch.no_grad()
def object_maps(image, question, objs):
    """One forward pass -> {obj: (x,y)} and {obj: 24x24 map}. The model's OWN attention."""
    inputs = processor(images=image, text=PROMPT_TEMPLATE.format(q=question),
                       return_tensors="pt").to(PRIMARY_DEVICE)
    out = model(**inputs, output_attentions=True)
    attn = out.attentions
    ids = inputs["input_ids"][0].tolist()
    vis_idx = torch.tensor([i for i, t in enumerate(ids) if t == IMAGE_TOKEN_INDEX],
                           device=attn[0].device)
    img_end = int(vis_idx.max()) + 1
    text_pos = list(range(img_end, len(ids)))
    base = _baseline_map(attn, text_pos, vis_idx, LOC_KW["layers"]) if LOC_KW["rel_att"] else None
    phrases = _relation_phrases(question)
    pos, maps = {}, {}
    for obj in objs:
        ph = _match_phrase(obj, phrases) if phrases else obj
        p = find_noun_positions(ids, ph, processor.tokenizer, img_end)
        if not p:
            pos[obj], maps[obj] = None, None
            continue
        mm = _noun_maps(attn, p, vis_idx, LOC_KW["layers"])
        m = mm.mean(0)
        if LOC_KW["rel_att"]:
            m = np.clip(m - base, 0, None)
        m = _denoise(m, LOC_KW["denoise"])
        cx, cy = _coords(m)
        pos[obj] = (cx, cy) if cx is not None else None
        maps[obj] = m
    del out, attn
    torch.cuda.empty_cache()
    return pos, maps

# --------------------------------------------------------- text deliveries --
def _region_phrase(cx, cy, gran=3):
    hx = "left" if cx < 0.40 else "right" if cx > 0.60 else "centre"
    vy = "bottom" if cy < 0.40 else "top" if cy > 0.60 else "middle"
    return "centre" if (hx == "centre" and vy == "middle") else f"{vy}-{hx}"

def _dist_word(d):
    return ("very close to" if d < 0.20 else "close to" if d < 0.40 else
            "a moderate distance from" if d < 0.65 else "far from")

def scaffold_nl_absolute(objs, pos):
    parts = [f"the {o} is situated in the {_region_phrase(*pos[o])} of the image"
             for o in objs if pos.get(o)]
    return ("In this image, " + " and ".join(parts) + ". ") if len(parts) >= 2 else None

def build_scaffold(pos, coord_scale=100):
    pts = [f"the {o} is at ({int(round(r[0]*coord_scale))}, {int(round(r[1]*coord_scale))})"
           for o, r in pos.items() if r]
    if len(pts) < 2:
        return None
    return (f"For the image, you will be given the (x, y) coordinates of each object in the "
            f"image where (0, 0) is the bottom-left corner and ({coord_scale}, {coord_scale}) "
            f"is the top-right corner; x increases to the right and y increases upward. So for "
            f"above/below questions, the object with higher y is the above object. For "
            f"right/left questions, the object with higher x is to the right"
            + "; ".join(pts) + ". ")

def spatial_brief(candidates, reference, pos):
    cand = [o for o in candidates if pos.get(o)]
    if len(cand) < 2:
        return None
    a, b = cand[0], cand[1]
    locs = [f"the {a} is in the {_region_phrase(*pos[a])}",
            f"the {b} is in the {_region_phrase(*pos[b])}"]
    has_ref = bool(reference and pos.get(reference))
    if has_ref:
        locs.append(f"the {reference} is in the {_region_phrase(*pos[reference])}")
    intro = "Based on a spatial analysis of the image: " + ", ".join(locs) + " of the image. "
    if has_ref:
        rx, ry = pos[reference]
        da = math.hypot(pos[a][0] - rx, pos[a][1] - ry)
        db = math.hypot(pos[b][0] - rx, pos[b][1] - ry)
        closer, farther = (a, b) if da < db else (b, a)
        body = (f"In the image plane, the {a} is {_dist_word(da)} the {reference}, and the {b} "
                f"is {_dist_word(db)} the {reference}. The {closer} is closer to the "
                f"{reference} than the {farther}. ")
    else:
        (ax, ay), (bx, by) = pos[a], pos[b]
        hor = (f"the {a} is to the left of the {b}" if ax < bx else
               f"the {a} is to the right of the {b}") if abs(ax - bx) > 0.1 else \
              f"{a} and {b} are at similar levels"
        ver = (f"the {a} is above the {b}" if ay > by else
               f"the {a} is below the {b}") if abs(ay - by) > 0.1 else \
              f"{a} and {b} are at similar heights"
        d = math.hypot(ax - bx, ay - by)
        sep = "far apart" if d > 0.5 else "a moderate distance apart" if d > 0.25 else "close together"
        body = f"Horizontally, {hor}. Vertically, {ver}. Overall, the two objects are {sep} in the image."
    return intro + body

@torch.no_grad()
def llava_generate(image, question):
    inputs = processor(images=image, text=PROMPT_TEMPLATE.format(q=question),
                       return_tensors="pt").to(PRIMARY_DEVICE)
    inputs["pixel_values"] = inputs["pixel_values"].to(DTYPE)
    n_in = inputs["input_ids"].shape[1]
    gen = model.generate(**inputs, max_new_tokens=ANS_MAX_NEW, do_sample=False, num_beams=1,
                         pad_token_id=processor.tokenizer.eos_token_id)
    return processor.tokenizer.decode(gen[0, n_in:], skip_special_tokens=True).strip()

# ------------------------------------------------------- latent steering -----
_RICH = ["on the far left of the image", "on the left side of the image",
         "on the right side of the image", "on the far right of the image",
         "at the very top of the image", "at the top of the image",
         "at the bottom of the image", "at the very bottom of the image"]

@torch.no_grad()
def _hidden_of(phrase, layer):
    inputs = processor(images=Image.new("RGB", (336, 336)),
                       text=PROMPT_TEMPLATE.format(q=phrase), return_tensors="pt").to(PRIMARY_DEVICE)
    inputs["pixel_values"] = inputs["pixel_values"].to(DTYPE)
    out = model(**inputs, output_hidden_states=True)
    return out.hidden_states[layer][0, -1, :].float().cpu()

_DIRS = None
def steer_dirs():
    """(dx, dy) unit directions in activation space: 'right'-'left' and 'up'-'down'."""
    global _DIRS
    if _DIRS is not None:
        return _DIRS
    L = STEER["layer"]
    h = {p: _hidden_of(p, L) for p in _RICH}
    dx = ((h[_RICH[2]] + h[_RICH[3]]) - (h[_RICH[0]] + h[_RICH[1]])) / 2
    dy = ((h[_RICH[4]] + h[_RICH[5]]) - (h[_RICH[6]] + h[_RICH[7]])) / 2
    if STEER["orthogonalize"]:
        dy = dy - (dy @ dx) / (dx @ dx + 1e-8) * dx
    dx = dx / (dx.norm() + 1e-8)
    dy = dy / (dy.norm() + 1e-8)
    _DIRS = (dx, dy)
    return _DIRS

def _all_occurrences(ids, word, tokenizer, img_end):
    out, start = [], img_end
    while True:
        p = find_noun_positions(ids, word, tokenizer, start)
        if not p:
            return out
        out.extend(p)
        start = p[-1] + 1

def _make_steer_hook(pos2combo, beta):
    def hook(module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        if h.shape[1] == 1:
            return out
        for idx, combo in pos2combo.items():
            if idx < h.shape[1]:
                tok = h[:, idx, :]
                scale = tok.norm(dim=-1, keepdim=True)
                h[:, idx, :] = tok + beta * scale * combo.to(h.dtype).to(h.device)
        return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h
    return hook

@torch.no_grad()
def answer_steer_abs(image, question, objs, pos):
    dx, dy = steer_dirs()
    inputs = processor(images=image, text=PROMPT_TEMPLATE.format(q=question),
                       return_tensors="pt").to(PRIMARY_DEVICE)
    ids = inputs["input_ids"][0].tolist()
    img_end = max(i for i, t in enumerate(ids) if t == IMAGE_TOKEN_INDEX) + 1
    pos2combo = {}
    for o in objs:
        if not pos.get(o):
            continue
        cx, cy = pos[o]
        combo = (2 * cx - 1) * dx + (2 * cy - 1) * dy      # each token -> its OWN position
        for q in _all_occurrences(ids, o, processor.tokenizer, img_end):
            pos2combo[q] = combo
    handle = LM_LAYERS[STEER["layer"]].register_forward_hook(
        _make_steer_hook(pos2combo, STEER["beta_abs"]))
    try:
        return llava_generate(image, question)
    finally:
        handle.remove()

@torch.no_grad()
def answer_steer_rel(image, question, cands, pos):
    """One relational push applied equal-and-opposite — the pair driven apart."""
    dx, dy = steer_dirs()
    if not (pos.get(cands[0]) and pos.get(cands[1])):
        return None
    (ax, ay), (bx, by) = pos[cands[0]], pos[cands[1]]
    ddx, ddy = ax - bx, ay - by
    n = math.hypot(ddx, ddy) + 1e-8
    push = (ddx / n) * dx + (ddy / n) * dy
    inputs = processor(images=image, text=PROMPT_TEMPLATE.format(q=question),
                       return_tensors="pt").to(PRIMARY_DEVICE)
    ids = inputs["input_ids"][0].tolist()
    img_end = max(i for i, t in enumerate(ids) if t == IMAGE_TOKEN_INDEX) + 1
    pos2combo = {}
    for o, sign in ((cands[0], +1.0), (cands[1], -1.0)):
        for q in _all_occurrences(ids, o, processor.tokenizer, img_end):
            pos2combo[q] = sign * push
    handle = LM_LAYERS[STEER["layer"]].register_forward_hook(
        _make_steer_hook(pos2combo, STEER["beta_rel"]))
    try:
        return llava_generate(image, question)
    finally:
        handle.remove()

# ------------------------------------------- the relational binding interface -
def fourier_pos(coords, n_freq=6):
    freqs = (2.0 ** torch.arange(n_freq, dtype=coords.dtype, device=coords.device)) * math.pi
    xy = coords.unsqueeze(-1) * freqs.view(1, 1, n_freq)
    return torch.cat([torch.sin(xy), torch.cos(xy)], dim=-1).reshape(coords.shape[0], -1)

class RelationalAbstractor(nn.Module):
    """One relational-cross-attention block. Values are content-independent learned
    symbols (the relational bottleneck) -> the output can encode only how the objects
    relate, never what they are."""
    def __init__(self, hidden, *, max_obj=4, n_roles=4, n_freq=6, d_model=256,
                 n_heads=4, use_content=False, content_dim=None, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.hidden, self.max_obj = hidden, max_obj
        self.n_heads, self.d_model, self.d_head = n_heads, d_model, d_model // n_heads
        self.use_content = use_content
        self.pos_proj = nn.Linear(4 * n_freq, d_model)
        self.role_emb = nn.Embedding(n_roles, d_model)
        if use_content:
            self.content_proj = nn.Linear(content_dim, d_model)
        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.symbols = nn.Parameter(torch.randn(max_obj, d_model) * 0.02)
        self.drop = nn.Dropout(dropout)
        self.readout = nn.Parameter(torch.randn(1, d_model) * 0.02)
        self.out = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, hidden),
                                 nn.GELU(), nn.Linear(hidden, hidden))
        nn.init.zeros_(self.out[-1].weight); nn.init.zeros_(self.out[-1].bias)
        self.attn_scale = self.d_head ** -0.5
        self.pool_scale = d_model ** -0.5

    def forward(self, coords, roles, content=None):
        N = coords.shape[0]
        E = self.pos_proj(fourier_pos(coords)) + self.role_emb(roles)
        if self.use_content and content is not None:
            E = E + self.content_proj(content)
        q = self.Wq(E).view(N, self.n_heads, self.d_head).transpose(0, 1)
        k = self.Wk(E).view(N, self.n_heads, self.d_head).transpose(0, 1)
        R = torch.softmax((q @ k.transpose(-1, -2)) * self.attn_scale, dim=-1)
        S = self.symbols[:N].view(N, self.n_heads, self.d_head).transpose(0, 1)
        abstract = (R @ S).transpose(0, 1).reshape(N, self.d_model)
        abstract = self.drop(abstract)
        attn = torch.softmax((self.readout @ abstract.transpose(0, 1)) * self.pool_scale, dim=-1)
        return self.out((attn @ abstract).squeeze(0))

class AbstractorController:
    """Writes the relation token into the residual stream at a reserved slot.
    The slot is a duplicate of the final prompt token, so no content is displaced."""
    def __init__(self, module, layer_idx, inject_beta=1.0):
        self.module, self.layer_idx = module, int(layer_idx)
        self.inject_beta = float(inject_beta)
        self.active = False
        self.coords = self.roles = None
        self.sentinel_mask = self.prefill_len = None
        self.dev = next(module.parameters()).device
        self._handle = None

    def set_example(self, coords, roles, prefill_len):
        self.coords = torch.as_tensor(coords, dtype=torch.float32, device=self.dev)
        self.roles = torch.as_tensor(roles, dtype=torch.long, device=self.dev)
        self.prefill_len = int(prefill_len)
        m = torch.zeros(self.prefill_len, dtype=torch.float32, device=self.dev)
        m[self.prefill_len - 1] = 1.0
        self.sentinel_mask = m.view(1, self.prefill_len, 1)
        self.active = True

    def clear(self):
        self.active = False

    def _prehook(self, module, args, kwargs):
        if not self.active:
            return None
        in_kwargs = "hidden_states" in kwargs
        hs = kwargs["hidden_states"] if in_kwargs else args[0]
        if hs.shape[1] != self.prefill_len:      # prefill only; decode reuses the cache
            return None
        r = self.module(self.coords, self.roles)
        sent_norm = hs[0, self.prefill_len - 1, :].detach().float().norm()
        rn = r.norm().clamp_min(1e-6)
        r = r * ((self.inject_beta * sent_norm) / rn).clamp(max=1.0)   # scale DOWN only
        r = r.to(dtype=hs.dtype, device=hs.device)
        mask = self.sentinel_mask.to(dtype=hs.dtype, device=hs.device)
        hs = hs + mask * r.view(1, 1, -1)
        if in_kwargs:
            kwargs = dict(kwargs); kwargs["hidden_states"] = hs
            return args, kwargs
        return (hs,) + tuple(args[1:]), kwargs

    def install(self):
        self.remove()
        self._handle = LM_LAYERS[self.layer_idx].register_forward_pre_hook(
            self._prehook, with_kwargs=True)
        return self

    def remove(self):
        if self._handle is not None:
            self._handle.remove(); self._handle = None

print(f"Fetching interface weights from {HF_REPO}…")
_ckpt_path = hf_hub_download(HF_REPO, CKPT_FILE, repo_type="model")
_ck = torch.load(_ckpt_path, map_location="cpu")
_cfg = _ck.get("cfg", {})
L_INJECT = int(_ck.get("L_INJECT", 14))
_dev_inject = next(LM_LAYERS[L_INJECT].parameters()).device
ABS = RelationalAbstractor(
    _cfg.get("hidden", HIDDEN), max_obj=_cfg.get("max_obj", 4),
    n_heads=_cfg.get("n_heads", 4), d_model=_cfg.get("d_model", 256),
    n_freq=_cfg.get("n_freq", 6), use_content=_cfg.get("use_content", False),
    content_dim=_cfg.get("content_dim")).to(_dev_inject)
ABS.load_state_dict(_ck["state_dict"])
ABS.eval()
ABS_CTRL = AbstractorController(ABS, L_INJECT,
                                inject_beta=float(_ck.get("inject_beta", 1.0))).install()
ABS_CTRL.clear()
_n_par = sum(p.numel() for p in ABS.parameters())
_n_bb = sum(p.numel() for p in model.parameters())
print(f"interface: {_n_par:,} params @ layer {L_INJECT} "
      f"({100*_n_par/_n_bb:.3f}% of the {_n_bb/1e9:.2f}B backbone) | beta={ABS_CTRL.inject_beta}")

# swap-asymmetry self-test: the token MUST change when the objects swap
with torch.no_grad():
    _a = ABS(torch.tensor([[0.2, 0.5], [0.8, 0.5]], device=_dev_inject),
             torch.tensor([0, 1], device=_dev_inject))
    _b = ABS(torch.tensor([[0.8, 0.5], [0.2, 0.5]], device=_dev_inject),
             torch.tensor([0, 1], device=_dev_inject))
    assert not torch.allclose(_a, _b), "loaded token is swap-symmetric — wrong checkpoint?"
print("self-test OK: relation token is swap-asymmetric.")

def _build_prompt_ids(view):
    enc = processor(images=view["image"], text=PROMPT_TEMPLATE.format(q=view["question"]),
                    return_tensors="pt")
    ids = enc["input_ids"].to(PRIMARY_DEVICE)
    am = enc["attention_mask"].to(PRIMARY_DEVICE)
    pv = enc["pixel_values"].to(PRIMARY_DEVICE, dtype=DTYPE)
    ids = torch.cat([ids, ids[:, -1:].clone()], dim=1)   # the reserved slot
    am = torch.cat([am, am[:, -1:].clone()], dim=1)
    return ids, am, pv

@torch.no_grad()
def answer_abstractor(view, coords, roles, swap=False):
    if swap:
        coords = [list(c) for c in coords]
        coords[0], coords[1] = coords[1], coords[0]
    ids, am, pv = _build_prompt_ids(view)
    ABS_CTRL.set_example(coords, roles, ids.shape[1])
    try:
        gen = model.generate(input_ids=ids, attention_mask=am, pixel_values=pv,
                             max_new_tokens=ANS_MAX_NEW, do_sample=False, num_beams=1,
                             pad_token_id=processor.tokenizer.eos_token_id)
    finally:
        ABS_CTRL.clear()
    return processor.tokenizer.decode(gen[0, ids.shape[1]:], skip_special_tokens=True).strip()

# ------------------------------------------------------------- MCQ scoring ---
def _gt_letter_index(gt, choices):
    s = str(gt).strip()
    m = re.match(r"^\(?([A-J])\)?$", s, re.I)
    if m:
        return LETTERS.index(m.group(1).upper())
    for k, c in enumerate(choices):
        if str(c).strip().lower() == s.lower():
            return k
    return -1

# Mutually-exclusive families. Naming >1 distinct member of the same family is
# hedging across exclusive options -> WRONG (no first-mention luck), matching the
# notebook's score_mcq. Each set lists the surface forms that count as that option.
_SYN = [{"left", "on the left", "to the left", "left side", "left of"},
        {"right", "on the right", "to the right", "right side", "right of"},
        {"above", "over", "on top of", "top", "higher"},
        {"below", "under", "underneath", "beneath", "bottom", "lower"}]

def _norm(t):
    t = re.sub(r"[^a-z0-9 ]", " ", str(t).lower())
    return re.sub(r"\s+", " ", t).strip()

def _forms(choice):
    """Every surface form that should count as this choice."""
    c = _norm(choice)
    for fam in _SYN:
        if c in fam:
            return fam
    return {c}

def score(model_out, choices, gt):
    """Letter first, then whole-word surface match over each option's synonyms.
    Hedging across exclusive options -> wrong. Unparseable -> abstain (None)."""
    if model_out is None:
        return None
    gi = _gt_letter_index(gt, choices)
    if gi < 0:
        return None
    out = str(model_out).strip()
    m = re.match(r"^\(?([A-J])\)?[\.\):,]?(?:\s|$)", out, re.I)
    if m:
        k = LETTERS.index(m.group(1).upper())
        if k < len(choices):
            return k == gi
    text = _norm(out)
    hits = set()
    for k, c in enumerate(choices):
        for form in _forms(c):
            if re.search(rf"\b{re.escape(form)}\b", text):
                hits.add(k)
                break
    if len(hits) == 1:
        return hits.pop() == gi
    if len(hits) > 1:
        return False          # named two exclusive options -> hedge -> wrong
    return None               # nothing parseable -> abstain

def _mark(ok):
    return "✅" if ok is True else "❌" if ok is False else "—"

# ------------------------------------------------------------------ figure ---
def _overlay_figure(view, objs, pos, maps):
    n = len(objs)
    fig, axes = plt.subplots(1, n + 1, figsize=(4.2 * (n + 1), 4.4))
    img = view["image"].convert("RGB")
    axes[0].imshow(img); axes[0].set_title("input", fontsize=11)
    axes[0].axis("off")
    for ax, o in zip(axes[1:], objs):
        ax.imshow(img.resize((336, 336)), alpha=0.55)
        if maps.get(o) is not None:
            m = maps[o]
            mm = (m - m.min()) / (np.ptp(m) + 1e-8)
            ax.imshow(np.kron(mm, np.ones((14, 14))), cmap="jet", alpha=0.55)
        if pos.get(o):
            cx, cy = pos[o]
            ax.scatter([cx * 335], [(1 - cy) * 335], s=260, marker="o",
                       facecolors="none", edgecolors="white", linewidths=2.6)
            ax.set_title(f"{o}\n({cx:.2f}, {cy:.2f})", fontsize=11)
        else:
            ax.set_title(f"{o}\n(not localised)", fontsize=11, color="crimson")
        ax.axis("off")
    fig.suptitle("The frozen model's own attention — this is where it is already looking",
                 fontsize=12, y=1.02)
    fig.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf)

# --------------------------------------------------------------- the demo ----
_HDR = ("| Condition | Delivery | Answer | |\n"
        "|---|---|---|---|\n")

def _row(name, kind, ans, ok):
    a = (str(ans).replace("|", "\\|").replace("\n", " ")[:60]) if ans else "_(n/a)_"
    return f"| **{name}** | {kind} | `{a}` | {_mark(ok)} |\n"

def run(index, progress=gr.Progress()):
    """Streams: figure + a table that grows one row per condition."""
    try:
        idx = int(index)
    except (TypeError, ValueError):
        yield None, "### ⚠️ Enter a whole number.", ""
        return
    if not (0 <= idx < len(cvbench)):
        yield None, f"### ⚠️ Index out of range (0–{len(cvbench)-1}).", ""
        return

    v = _view(idx)
    if v["task"] != "Relation":
        yield None, (f"### ⚠️ Item {idx} is a **{v['task']}** item.\n\n"
                     f"The interface is gated to image-plane relations and the router "
                     f"would not fire here. Try one of: "
                     f"`{', '.join(str(i) for i in REL_IDX[:8])}` …"), ""
        return

    gt_i = _gt_letter_index(v["answer"], v["choices"])
    gt_txt = v["choices"][gt_i] if gt_i >= 0 else str(v["answer"])
    split = _split_of(REL_IDX.index(idx)) if idx in REL_IDX else None
    banner = ""
    if split == "train":
        banner = ("> ⚠️ **This item is in the interface's training split.** "
                  "The headline +9.9 is measured on the held-out test split only.\n\n")
    elif split:
        banner = f"> Split: **{split}**\n\n"

    info = (f"{banner}### Item {idx} — CV-Bench Relation\n\n"
            f"**Q:** {v['clean_question']}\n\n"
            f"**Options:** {' · '.join(str(c) for c in v['choices'])}  \n"
            f"**Ground truth:** `{gt_txt}`\n")

    # ---- localise -----------------------------------------------------------
    progress(0.05, desc="Reading the model's attention…")
    yield None, info + "\n_Localising objects from the model's own attention…_", ""

    objs = _relation_objects(v["clean_question"], v["target_class"])[:2]
    if len(objs) < 2:
        yield None, info + "\n### ⚠️ Could not parse two objects from this question.", ""
        return
    pos, maps = object_maps(v["image"], v["question"], objs)
    fig = _overlay_figure(v, objs, pos, maps)

    if not (pos.get(objs[0]) and pos.get(objs[1])):
        yield fig, info + ("\n### ⚠️ Localisation failed for at least one object.\n\n"
                           "Every condition falls back to vanilla here, so there is "
                           "nothing to compare. Try another index."), ""
        return

    coords = [[float(pos[o][0]), float(pos[o][1])] for o in objs]
    roles = list(range(len(objs)))

    oracle = None
    (ax_, ay_), (bx_, by_) = pos[objs[0]], pos[objs[1]]
    opts = [str(c).strip().lower() for c in v["choices"]]
    if any(o in ("left", "right") for o in opts):
        oracle = "left" if ax_ < bx_ else "right"
    elif any(o in ("above", "below") for o in opts):
        oracle = "above" if ay_ > by_ else "below"
    coord_note = (f"\n**Coordinates read from attention:** "
                  f"`{objs[0]}` = ({coords[0][0]:.2f}, {coords[0][1]:.2f}) · "
                  f"`{objs[1]}` = ({coords[1][0]:.2f}, {coords[1][1]:.2f})"
                  + (f"  \n**These two coordinates alone imply:** `{oracle}`" if oracle else "")
                  + "\n")
    info = info + coord_note

    table = _HDR
    yield fig, info, table + "_Running conditions…_"

    # ---- conditions, streamed in order --------------------------------------
    steps = [
        ("vanilla",      "frozen model, no help",
         lambda: llava_generate(v["image"], v["question"])),
        ("nl_absolute",  "each position in words",
         lambda: (lambda s: llava_generate(v["image"], s + "\n" + v["question"]) if s else None)(
             scaffold_nl_absolute(objs, pos))),
        ("coords_xy",    "raw (x, y) as text",
         lambda: (lambda s: llava_generate(v["image"], s + "\n" + v["question"]) if s else None)(
             build_scaffold({o: pos[o] for o in objs if pos.get(o)}))),
        ("steer_abs",    "position vector per token",
         lambda: answer_steer_abs(v["image"], v["question"], objs, pos)),
        ("steer_rel",    "contrastive push on the pair",
         lambda: answer_steer_rel(v["image"], v["question"], objs, pos)),
        ("brief_2d",     "the relation, stated (teacher)",
         lambda: (lambda s: llava_generate(v["image"], s + "\n" + v["question"]) if s else None)(
             spatial_brief(objs, None, pos))),
        ("abstractor",   "**relation token → residual stream**",
         lambda: answer_abstractor(v, coords, roles)),
        ("abstractor (swapped)", "control: coordinates exchanged",
         lambda: answer_abstractor(v, coords, roles, swap=True)),
    ]

    results = {}
    for k, (name, kind, fn) in enumerate(steps):
        progress((k + 1) / (len(steps) + 1), desc=f"{name}…")
        try:
            ans = fn()
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            ans = None
        except Exception as e:
            ans = f"[error: {type(e).__name__}]"
        ok = score(ans, v["choices"], v["answer"]) if not str(ans).startswith("[error") else None
        results[name] = ok
        table += _row(name, kind, ans, ok)
        if name == "brief_2d":
            table += "| | | | |\n"          # visual break before our method
        yield fig, info, table

    # ---- the reading --------------------------------------------------------
    van, abst, swp = results.get("vanilla"), results.get("abstractor"), results.get("abstractor (swapped)")
    notes = ["\n---\n"]
    if van is False and abst is True:
        notes.append("**The interface fixed this item.** The frozen model got it backwards; "
                     "the same coordinates, delivered as a relation token, corrected it.\n")
    elif van is True and abst is True:
        notes.append("Both correct — the relation token did no harm.\n")
    elif abst is False:
        notes.append("The interface did not fix this one. Single items are noisy; "
                     "the +9.9-point claim is over the 131-item test split.\n")
    if abst is True and swp is False:
        notes.append("\n**Swap control passed.** Exchanging the two coordinates flipped the "
                     "answer, so the token carries the *relation* — not a per-object prior "
                     "and not a generic nudge. This is the check `steer_rel` fails at the "
                     "population level (Table 2).\n")
    elif abst is True and swp is True:
        notes.append("\n**Note:** the swap did not flip this item. On the test split the swap "
                     "drops accuracy to 29.8% (below chance) — but it is not guaranteed "
                     "per-item.\n")
    yield fig, info, table + "".join(notes)

# ------------------------------------------------------------------- UI ------
_EXAMPLES = [467, 343, 451]
             
with gr.Blocks(title="Relations Need a Home", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# Relations Need a Home\n"
        "### A Relational Binding Interface for Spatial Reasoning in Frozen VLMs\n"
        "Haziq Mohammad Khalid · Hamad Siddiqi - COE486 Final Project\n\n"
        "Pick a CV-Bench **Relation** index. You'll see where the frozen LLaVA-1.5-7B is "
        "already looking, then the same coordinates delivered five different ways, and "
        "finally as a single **relation token** written into the residual stream. "
        "The backbone is never fine-tuned.")
    with gr.Row():
        idx_in = gr.Number(label="CV-Bench index", value=REL_IDX[0], precision=0, scale=3)
        go = gr.Button("Run", variant="primary", scale=1)
    gr.Markdown(f"_Relation items live at indices "
                f"{REL_IDX[0]}–{REL_IDX[-1]} ({len(REL_IDX)} total)._")
    fig_out = gr.Image(label="Attention maps + localised coordinates", type="pil")
    info_out = gr.Markdown()
    tbl_out = gr.Markdown()
    gr.Examples(_EXAMPLES, inputs=[idx_in], label="Try one")
    go.click(run, inputs=[idx_in], outputs=[fig_out, info_out, tbl_out])
    idx_in.submit(run, inputs=[idx_in], outputs=[fig_out, info_out, tbl_out])

if __name__ == "__main__":
    demo.queue().launch(share=True, debug=False)
else:
    demo.queue().launch(share=True, inline=True)

GPU: Tesla T4, Tesla T4  |  total VRAM ~31 GB
Loading LLaVA-1.5-7B (frozen)…


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

...

CV-Bench: 2638 rows | 650 Relation items


relation_split.json: 0.00B [00:00, ?B/s]

split loaded: train=454 val=65 test=131
Fetching interface weights from Haziq-exe/relational-binding-interface…


abstractor_relation_kl_ce.pt:   0%|          | 0.00/71.9M [00:00<?, ?B/s]

interface: 17,974,272 params @ layer 14 (0.254% of the 7.06B backbone) | beta=1.0
self-test OK: relation token is swap-asymmetric.
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://64cb64cbcd6eaa3949.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
